# Hospital Patient Queue Management System
A command-line application that manages patient queues for hospital specializations, built from three classes:

| Class | Responsibility |
|---|---|
| `Patient` | `name` and `status` (0 = normal, 1 = urgent, 2 = super-urgent) with string / status formatting |
| `Specialization` | One waiting queue (limited capacity): add patients by urgency, get next, remove by name, capacity check |
| `OperationsManager` | The user interface: add, list, get next, remove, end program |

**Queue rule:** patients are ordered by urgency – super-urgent first, then urgent, then normal. Patients with the *same* status are served first-come-first-served.

## 1. `Patient` class

In [1]:
class Patient:
    STATUS_NAMES = {0: "normal", 1: "urgent", 2: "super urgent"}

    def __init__(self, name, status):
        if status not in self.STATUS_NAMES:
            raise ValueError("Status must be 0 (normal), 1 (urgent) or 2 (super urgent)")
        self.name = name
        self.status = status

    def status_text(self):
        return self.STATUS_NAMES[self.status]

    def __str__(self):
        return f"{self.name} ({self.status_text()})"

print(Patient("Anita", 2))
print(Patient("Rahul", 0))

Anita (super urgent)
Rahul (normal)


## 2. `Specialization` class
* `add_patient(name, status)` – inserts in the right place; returns `False` if the queue is full.
* `get_next_patient()` – removes and returns the patient at the front.
* `remove_patient(name)` – removes a patient who leaves before being served.
* `is_full()` – capacity check.

In [2]:
class Specialization:
    MAX_QUEUE = 5

    def __init__(self, name, max_queue=MAX_QUEUE):
        self.name = name
        self.max_queue = max_queue
        self.queue = []                       # front of the line is index 0

    def is_full(self):
        return len(self.queue) >= self.max_queue

    def add_patient(self, name, status):
        if self.is_full():
            return False
        new_patient = Patient(name, status)
        # insert after the last patient whose urgency is >= the new patient's urgency
        position = len(self.queue)
        for i, p in enumerate(self.queue):
            if p.status < new_patient.status:
                position = i
                break
        self.queue.insert(position, new_patient)
        return True

    def get_next_patient(self):
        return self.queue.pop(0) if self.queue else None

    def remove_patient(self, name):
        for i, p in enumerate(self.queue):
            if p.name.lower() == name.lower():
                return self.queue.pop(i)
        return None

    def display(self):
        if not self.queue:
            print(f"  [{self.name}] no patients")
        else:
            print(f"  [{self.name}] {len(self.queue)}/{self.max_queue} patients:")
            for pos, p in enumerate(self.queue, 1):
                print(f"     {pos}. {p}")

# quick test
cardio = Specialization("Cardiology", max_queue=3)
cardio.add_patient("Ravi", 0)
cardio.add_patient("Meena", 1)
cardio.add_patient("Asha", 2)            # goes to the very front
print("Full?", cardio.is_full(), "| Add 4th:", cardio.add_patient("Karan", 0))
cardio.display()
print("Next:", cardio.get_next_patient())
cardio.display()

Full? True | Add 4th: False
  [Cardiology] 3/3 patients:
     1. Asha (super urgent)
     2. Meena (urgent)
     3. Ravi (normal)
Next: Asha (super urgent)
  [Cardiology] 2/3 patients:
     1. Meena (urgent)
     2. Ravi (normal)


## 3. `OperationsManager` class
A menu-driven interface that holds several specializations. Input is validated at every step.

In [3]:
class OperationsManager:
    def __init__(self, specializations=None, max_queue=5):
        names = specializations or ["Cardiology", "Dermatology", "Orthopedics", "Pediatrics", "Neurology"]
        self.specializations = [Specialization(n, max_queue) for n in names]

    # ---------- input helpers ----------
    def _read_int(self, prompt, low, high):
        while True:
            try:
                value = int(input(prompt))
                if low <= value <= high:
                    return value
                print(f"Please enter a number between {low} and {high}.")
            except ValueError:
                print("Please enter a whole number.")

    def _choose_specialization(self):
        print("Specializations:")
        for i, s in enumerate(self.specializations, 1):
            print(f"  {i}) {s.name}")
        idx = self._read_int("Enter specialization number: ", 1, len(self.specializations))
        return self.specializations[idx - 1]

    def print_menu(self):
        print("\n===== Hospital Queue System =====")
        print("1) Add new patient")
        print("2) Print all patients")
        print("3) Get next patient")
        print("4) Remove a leaving patient")
        print("5) End the program")

    # ---------- actions ----------
    def add_patient(self):
        spec = self._choose_specialization()
        name = input("Enter patient name: ").strip()
        status = self._read_int("Status (0 = normal, 1 = urgent, 2 = super urgent): ", 0, 2)
        if spec.add_patient(name, status):
            print(f"{name} added to {spec.name}.")
        else:
            print(f"Sorry, the {spec.name} queue is full ({spec.max_queue} patients). Try again later.")

    def print_all(self):
        print("\nAll waiting patients:")
        if all(not s.queue for s in self.specializations):
            print("  No patients at all.")
        for s in self.specializations:
            if s.queue:
                s.display()

    def get_next(self):
        spec = self._choose_specialization()
        patient = spec.get_next_patient()
        if patient:
            print(f"{patient.name}, please go to the doctor ({spec.name}).")
        else:
            print(f"No patients are waiting in {spec.name}. Have a rest, Dr!")

    def remove_patient(self):
        spec = self._choose_specialization()
        name = input("Enter name of the patient to remove: ").strip()
        removed = spec.remove_patient(name)
        print(f"{removed.name} removed from {spec.name}." if removed
              else f"No patient named '{name}' in {spec.name}.")

    # ---------- main loop ----------
    def run(self):
        actions = {1: self.add_patient, 2: self.print_all, 3: self.get_next, 4: self.remove_patient}
        while True:
            self.print_menu()
            choice = self._read_int("Enter your choice (1-5): ", 1, 5)
            if choice == 5:
                print("Program ended. Goodbye!")
                break
            actions[choice]()

## 4. Demo run with scripted input
Scenario in **Cardiology** (queue size 3 for the demo): add Ravi (normal), Meena (urgent), Asha (super urgent) → a 4th patient is rejected (queue full) → list → serve next (Asha) → remove Meena → list → next patient from an empty specialization → exit.

In [4]:
from unittest.mock import patch

def run_with_inputs(func, inputs):
    it = iter(inputs)
    def fake_input(prompt=""):
        value = next(it)
        print(f"{prompt}{value}")
        return value
    with patch("builtins.input", fake_input):
        return func()

script = [
    "1", "1", "Ravi", "0",
    "1", "1", "Meena", "1",
    "1", "1", "Asha", "2",
    "1", "1", "Karan", "0",       # rejected: queue full
    "2",
    "3", "1",                     # next patient -> Asha (super urgent first)
    "4", "1", "meena",            # Meena leaves
    "2",
    "3", "2",                     # Dermatology is empty
    "7",                          # invalid menu choice
    "5",
]
demo = OperationsManager(max_queue=3)
run_with_inputs(demo.run, script)


===== Hospital Queue System =====
1) Add new patient
2) Print all patients
3) Get next patient
4) Remove a leaving patient
5) End the program
Enter your choice (1-5): 1
Specializations:
  1) Cardiology
  2) Dermatology
  3) Orthopedics
  4) Pediatrics
  5) Neurology
Enter specialization number: 1
Enter patient name: Ravi
Status (0 = normal, 1 = urgent, 2 = super urgent): 0
Ravi added to Cardiology.

===== Hospital Queue System =====
1) Add new patient
2) Print all patients
3) Get next patient
4) Remove a leaving patient
5) End the program
Enter your choice (1-5): 1
Specializations:
  1) Cardiology
  2) Dermatology
  3) Orthopedics
  4) Pediatrics
  5) Neurology
Enter specialization number: 1
Enter patient name: Meena
Status (0 = normal, 1 = urgent, 2 = super urgent): 1
Meena added to Cardiology.

===== Hospital Queue System =====
1) Add new patient
2) Print all patients
3) Get next patient
4) Remove a leaving patient
5) End the program
Enter your choice (1-5): 1
Specializations:
  1) 

## 5. Run it interactively
Uncomment to use the real menu.

In [5]:
# OperationsManager().run()